# Silver to Gold (cópia do Alisson)

Transformação das tabelas `v_energy.silver` para `v_energy.gold`, uma seção por tabela do contrato
(`contracts/gold_schema.yaml`, v0.3). Pela DEC-011, Alisson, Arthur e Gabriel fazem cada um a sua cópia; o time
compara pontos fortes e fracos das três para montar a versão final. Este notebook é a cópia do Alisson.

**Como executar**
- `Run All` processa as tabelas em sequência. Com `gravar = nao` (padrão), cada tabela é montada e **validada**, sem gravar nada.
- Com `gravar = sim`, cada tabela só é gravada (Delta, `overwrite`) depois de passar em todas as travas, e a contagem gravada é conferida.
- Uma tabela que falha não é gravada; a seção **Resumo** falha no fim se alguma tabela falhou ou não foi executada.
- Ordem obrigatória: `gold_dim_uc` antes de `gold_fato_faturamento_mensal` (chave estrangeira lógica).

**Regras comuns da Gold**
- Nomes, ordem e tipos das colunas iguais ao contrato (`CONTRATO`, seção 0). Coluna a mais ou a menos é erro.
- Toda coluna `nullable: false` é conferida; domínios (`allowed`) e formatos (`pattern`) também.
- Nenhuma conversão de tipo perde valor em silêncio (`try_cast` conferido contra a origem).
- Cálculos de negócio ficam aqui (DEC/FEC, kWh faturado, situação da fatura); o portal e o agente só leem.

**Confidencial:** as saídas das células têm contagens derivadas do dataset. Limpe os outputs antes de exportar ou commitar.

## 0. Configuração
Execute esta célula antes de qualquer tabela.

In [0]:
import traceback
from datetime import datetime

from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.dropdown("gravar", "nao", ["nao", "sim"], "Gravar na Gold?")
GRAVAR = dbutils.widgets.get("gravar") == "sim"

CATALOG = "v_energy"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold_alisson"  # cópia do Alisson (DEC-011); v_energy.gold fica para a versão final
TABELAS_ESPERADAS = ["gold_dim_uc", "gold_fato_faturamento_mensal", "gold_reconciliacao_faturamento",
                     "gold_continuidade_alimentador_mensal", "gold_kpis_segmento_mensal", "gold_kpis_mensais",
                     "gold_receita_decomposicao", "gold_fila_suspeitas", "gold_medidor_uc", "gold_interrupcoes_uc",
                     "gold_atendimentos_uc", "gold_inspecoes_uc"]

if GRAVAR:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

RESULTADOS = {}  # status de cada tabela, usado no Resumo
GOLD_DF = {}  # DataFrames montados nesta execução (as tabelas seguintes usam daqui)
CACHE = {}  # cálculos usados por mais de uma tabela (ex.: kWh faturado)
print(f"Silver: {SILVER} | Gold: {GOLD} | gravar = {'sim' if GRAVAR else 'nao'}")


# ---------- Contrato da Gold (espelho de contracts/gold_schema.yaml v0.3) ----------

TIPOS = {"text": "string", "integer": "int", "boolean": "boolean", "date": "date", "timestamp": "timestamp",
         "money": "decimal(14,2)", "energy_kwh": "decimal(16,3)", "ratio": "double",
         "text_array": "array<string>"}
COMPETENCIA = r"^\d{4}-(0[1-9]|1[0-2])$"
TIPOS_MEDIDOR = ["eletromecanico", "eletronico", "smart"]
ACRESCIMO_BANDEIRA = {"verde": 0.0, "amarela": 0.01885, "vermelha_p1": 0.04463, "vermelha_p2": 0.07877}  # R$/kWh (DEC-029)


def col(tipo, nulo=False, dominio=None, padrao=None):
    return {"tipo": tipo, "nulo": nulo, "dominio": dominio, "padrao": padrao}


CONTRATO = {
    "gold_dim_uc": {
        "cod_uc": col("text"),
        "municipio": col("text"),
        "bairro": col("text"),
        "classe": col("text", dominio=["B1", "B2", "B3", "A4"]),
        "classe_cadastro": col("text", dominio=["Residencial", "Comercial", "Industrial", "Rural", "Poder Público"]),
        "cod_alimentador": col("text"),
        "possui_gd": col("boolean"),
        "consumo_medio_kwh_12m": col("energy_kwh", nulo=True),
        "titular_mascarado": col("text", nulo=True),
        "documento_mascarado": col("text", nulo=True),
        "potencia_kwp": col("ratio", nulo=True),
        "data_ligacao": col("date"),
        "data_conexao_gd": col("date", nulo=True),
        "modalidade_gd": col("text", nulo=True, dominio=["geracao_propria_uc", "autoconsumo_remoto", "geracao_compartilhada"]),
        "energia_injetada_kwh_mes": col("energy_kwh", nulo=True),
    },
    "gold_fato_faturamento_mensal": {
        "cod_uc": col("text"),
        "competencia": col("text", padrao=COMPETENCIA),
        "kwh_faturado": col("energy_kwh"),
        "kwh_leitura": col("energy_kwh"),
        "origem_kwh": col("text", dominio=["leitura", "leitura_atual_troca_medidor", "consumo_absoluto"]),
        "receita_faturada": col("money"),
        "receita_energia": col("money"),
        "tributos": col("money"),
        "receita_arrecadada": col("money", nulo=True),
        "tarifa_media": col("ratio", nulo=True),
        "bandeira": col("text", nulo=True, dominio=["verde", "amarela", "vermelha_p1", "vermelha_p2"]),
        "data_vencimento": col("date"),
        "data_pagamento": col("date", nulo=True),
        "status_pagamento": col("text", dominio=["paga", "paga_com_atraso", "em_aberto", "em_atraso"]),
        "dias_em_atraso": col("integer", nulo=True),
        "receita_bandeira": col("money"),
        "tipo_leitura": col("text", dominio=["real", "estimada"]),
        "tipo_medidor": col("text", dominio=TIPOS_MEDIDOR),
        "dias_faturados": col("integer", nulo=True),
    },
    "gold_reconciliacao_faturamento": {
        "competencia": col("text", padrao=COMPETENCIA),
        "receita_faturada_total": col("money"),
        "receita_faturada_orfas": col("money"),
        "receita_faturada_cadastro": col("money"),
        "tributos_cadastro": col("money"),
        "receita_energia_cadastro": col("money"),
        "n_faturas_orfas": col("integer"),
        "kwh_leitura_total": col("energy_kwh"),
        "kwh_correcao_troca_medidor": col("energy_kwh"),
        "kwh_correcao_sinal_invertido": col("energy_kwh"),
        "kwh_faturado_total": col("energy_kwh"),
    },
    "gold_continuidade_alimentador_mensal": {
        "cod_alimentador": col("text"),
        "competencia": col("text", padrao=COMPETENCIA),
        "dec": col("ratio"),
        "fec": col("ratio"),
        "n_uc": col("integer"),
        "n_interrupcoes": col("integer"),
        "horas_interrompidas": col("ratio"),
    },
    "gold_kpis_segmento_mensal": {
        "competencia": col("text", padrao=COMPETENCIA),
        "municipio": col("text"),
        "bairro": col("text"),
        "classe_cadastro": col("text", dominio=["Residencial", "Comercial", "Industrial", "Rural", "Poder Público"]),
        "classe": col("text", dominio=["B1", "B2", "B3", "A4"]),
        "possui_gd": col("boolean"),
        "cod_alimentador": col("text"),
        "tipo_medidor": col("text", dominio=TIPOS_MEDIDOR + ["sem_medidor"]),
        "n_uc": col("integer"),
        "kwh_faturado": col("energy_kwh"),
        "receita_faturada": col("money"),
        "receita_energia": col("money"),
        "tributos": col("money"),
        "receita_arrecadada": col("money", nulo=True),
        "horas_interrompidas": col("ratio"),
        "n_interrupcoes": col("integer"),
        "n_chamados": col("integer"),
        "soma_notas": col("ratio", nulo=True),
        "n_notas": col("integer"),
        "receita_bandeira": col("money"),
        "n_faturas": col("integer"),
        "n_faturas_nao_pagas": col("integer"),
        "valor_nao_pago": col("money"),
        "n_faturas_pagas_com_atraso": col("integer"),
        "soma_dias_em_atraso": col("integer"),
        "n_leituras": col("integer"),
        "n_leituras_estimadas": col("integer"),
    },
    "gold_receita_decomposicao": {
        "periodo_base": col("text", padrao=r"^\d{4}-\d{2}/\d{4}-\d{2}$"),
        "periodo_atual": col("text", padrao=r"^\d{4}-\d{2}/\d{4}-\d{2}$"),
        "municipio": col("text"),
        "bairro": col("text"),
        "classe": col("text", dominio=["B1", "B2", "B3", "A4"]),
        "possui_gd": col("boolean"),
        "receita_base": col("money"),
        "receita_atual": col("money"),
        "efeito_preco": col("money"),
        "efeito_tarifa": col("money"),
        "efeito_bandeira": col("money"),
        "efeito_volume": col("money"),
        "efeito_mix": col("money"),
    },
    "gold_fila_suspeitas": {
        "cod_uc": col("text"),
        "score_suspeita": col("integer"),
        "motivos": col("text_array"),
        "kwh_recuperavel_est": col("energy_kwh", nulo=True),
    },
    "gold_medidor_uc": {
        "cod_uc": col("text"),
        "cod_medidor": col("text"),
        "tipo_medidor": col("text", dominio=TIPOS_MEDIDOR),
        "modelo": col("text", nulo=True),
        "data_instalacao": col("date"),
        "idade_anos": col("ratio"),
        "n_trocas": col("integer"),
        "data_ultima_troca": col("date", nulo=True),
    },
    "gold_interrupcoes_uc": {
        "cod_ocorrencia": col("text"),
        "cod_uc": col("text"),
        "cod_alimentador": col("text"),
        "competencia": col("text", padrao=COMPETENCIA),
        "data_abertura": col("timestamp"),
        "data_encerramento": col("timestamp"),
        "duracao_horas": col("ratio"),
        "tipo": col("text"),
        "causa": col("text", nulo=True),
    },
    "gold_atendimentos_uc": {
        "protocolo": col("text"),
        "cod_uc": col("text"),
        "competencia": col("text", padrao=COMPETENCIA),
        "data_abertura": col("timestamp"),
        "data_fechamento": col("timestamp"),
        "canal": col("text"),
        "assunto": col("text"),
        "subassunto": col("text", nulo=True),
        "nota_satisfacao": col("ratio", nulo=True),
    },
    "gold_inspecoes_uc": {
        "cod_inspecao": col("text"),
        "cod_uc": col("text"),
        "data_inspecao": col("date"),
        "competencia": col("text", padrao=COMPETENCIA),
        "resultado": col("text", dominio=["irregularidade_confirmada", "sem_irregularidade"]),
        "tipo_irregularidade": col("text", nulo=True),
        "kwh_recuperado": col("energy_kwh"),
    },
    "gold_kpis_mensais": {
        "competencia": col("text", padrao=COMPETENCIA),
        "kwh_faturado": col("energy_kwh"),
        "receita_faturada": col("money"),
        "receita_energia": col("money"),
        "dec": col("ratio", nulo=True),
        "fec": col("ratio", nulo=True),
        "n_chamados": col("integer", nulo=True),
        "satisfacao_media": col("ratio", nulo=True),
    },
}


# ---------- Travas: se alguma falhar, a tabela não é gravada ----------

def exigir_zero(df, regras):
    """regras = {nome: condição}. Toda condição precisa ter 0 linhas."""
    r = df.select(*[
        F.sum(F.coalesce(cond, F.lit(False)).cast("int")).alias(nome) for nome, cond in regras.items()
    ]).first().asDict()
    violadas = {nome: n for nome, n in r.items() if n}
    assert not violadas, f"regras violadas (linhas): {violadas}"


def exigir_chave_unica(df, chave):
    total = df.count()
    unicos = df.select(*chave).distinct().count()
    assert total == unicos, f"chave {chave} duplicada: {total - unicos:,} linhas"
    return total


def exigir_mesma_contagem(**contagens):
    """Conferência de cardinalidade entre etapas (ex.: faturas = leituras = linhas do fato)."""
    assert len(set(contagens.values())) == 1, f"contagens diferentes: {contagens}"


def exigir_sem_orfaos(df, nome):
    orfaos = df.join(obter("gold_dim_uc").select("cod_uc"), "cod_uc", "left_anti").count()
    assert orfaos == 0, f"{nome}: {orfaos:,} linhas com cod_uc fora de gold_dim_uc"


def aplicar_contrato(df, tabela):
    """Nomes, ordem e tipos do contrato; confere nulidade, domínio, formato e perda na conversão."""
    contrato = CONTRATO[tabela]
    sobrando, faltando = set(df.columns) - set(contrato), set(contrato) - set(df.columns)
    assert not sobrando and not faltando, f"{tabela}: colunas diferentes do contrato. Sobrando: {sobrando} | Faltando: {faltando}"

    tipado = df.select(
        *[F.col(c).alias(f"_orig_{c}") for c in contrato],
        *[F.expr(f"try_cast(`{c}` AS {TIPOS[r['tipo']]})").alias(c) for c, r in contrato.items()])
    regras = {}
    for c, r in contrato.items():
        regras[f"{c}_perdido_na_conversao"] = F.col(c).isNull() & F.col(f"_orig_{c}").isNotNull()
        if not r["nulo"]:
            regras[f"{c}_nulo"] = F.col(c).isNull()
        if r["dominio"]:
            regras[f"{c}_fora_do_dominio"] = ~F.col(c).isin(*r["dominio"])
        if r["padrao"]:
            regras[f"{c}_fora_do_formato"] = ~F.col(c).rlike(r["padrao"])
    exigir_zero(tipado, regras)
    return tipado.select(*contrato)


def contar_por(df, coluna):
    return {f"{coluna}={r[coluna]}": r["count"] for r in df.groupBy(coluna).count().collect()}


# ---------- Execução ----------

def obter(nome):
    """DataFrame montado nesta execução; se a seção não rodou agora, lê a tabela Gold já gravada."""
    return GOLD_DF[nome] if nome in GOLD_DF else spark.table(f"{GOLD}.{nome}")


def executar(nome, construir):
    """Monta e valida; só grava com gravar = sim e depois de passar em todas as travas."""
    print(f"=== {nome}")
    try:
        df, info = construir()
        GOLD_DF[nome] = df
        if GRAVAR:
            destino = f"{GOLD}.{nome}"
            df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
            gravadas = spark.table(destino).count()
            assert gravadas == info["linhas"], f"gravadas {gravadas:,} != validadas {info['linhas']:,}"
            GOLD_DF[nome] = spark.table(destino)
            status = "gravada e conferida"
        else:
            status = "validada (sem gravar)"
        RESULTADOS[nome] = {"status": status, **info}
        print(f"[OK] {nome}: {status} | {info}")
    except Exception as e:
        RESULTADOS[nome] = {"status": "FALHA", "erro": f"{type(e).__name__}: {e}"}
        print(f"[FALHA] {nome}: nada foi gravado.")
        traceback.print_exc()

## 1. Energia faturada (`kwh_faturado`)
Cálculo usado pela `gold_dim_uc` (consumo médio) e pelo fato. Regra validada contra a fatura (DEC-023):

| Caso | `kwh_faturado` | `origem_kwh` |
|---|---|---|
| `consumo_kwh` não negativo | `consumo_kwh` | `leitura` |
| negativo, com medidor da UC instalado no mês da competência (troca de medidor) | `leitura_atual` | `leitura_atual_troca_medidor` |
| negativo, sem medidor novo no mês (sinal invertido) | `abs(consumo_kwh)` | `consumo_absoluto` |

Conferido no dado real: a regra reproduz os dois grupos do diagnóstico (troca de medidor e sinal invertido), exclusivos e completos.
O consumo da leitura já é líquido de GD (DEC-023): não há correção por geração própria.

Junto com a energia saem, por leitura, o tipo da leitura (real ou estimada), o tipo do medidor que a fez e os dias desde a leitura
anterior da UC (`dias_faturados`, vazio na primeira).

**Travas:** nenhuma leitura sem consumo; toda troca tem leitura regressiva ou sem leitura anterior; todo sinal invertido tem a leitura
atual maior que a anterior e `consumo_kwh = -(atual - anterior)` com tolerância de 1 kWh (arredondamento da origem, o mesmo piso do
LEI-07), ou não tem leitura anterior para conferir; nenhum `kwh_faturado` negativo; todo medidor de leitura existe no cadastro de
medidores; dias entre leituras positivos. Se uma trava falhar, a célula mostra a quebra das linhas fora do padrão antes de parar.

In [0]:
def energia_faturada():
    """kwh_faturado por UC × competência, calculado uma vez por execução."""
    if "energia" in CACHE:
        return CACHE["energia"]

    leituras = spark.table(f"{SILVER}.leituras").select("id_uc", "mes_referencia", "id_medidor", "data_leitura", "tipo_leitura",
                                                        "leitura_anterior", "leitura_atual", "consumo_kwh")
    tipo_medidor = spark.table(f"{SILVER}.medidores").select("id_medidor", F.lower("tipo").alias("tipo_medidor"))
    leituras = (leituras.join(tipo_medidor, "id_medidor", "left")
        .withColumn("dias_faturados", F.datediff("data_leitura", F.lag("data_leitura").over(
            Window.partitionBy("id_uc").orderBy("mes_referencia")))))
    instalacoes = (spark.table(f"{SILVER}.medidores")
        .select("id_uc", F.date_format("data_instalacao", "yyyy-MM").alias("mes_referencia"))
        .dropna().distinct().withColumn("_troca", F.lit(True)))
    leituras = leituras.join(instalacoes, ["id_uc", "mes_referencia"], "left").fillna(False, ["_troca"])

    negativo = F.col("consumo_kwh") < 0
    leituras = (leituras
        .withColumn("origem_kwh", F.when(~negativo, "leitura")
                                   .when(F.col("_troca"), "leitura_atual_troca_medidor")
                                   .otherwise("consumo_absoluto"))
        .withColumn("kwh_faturado", F.when(~negativo, F.col("consumo_kwh"))
                                     .when(F.col("_troca"), F.col("leitura_atual"))
                                     .otherwise(F.abs("consumo_kwh"))))

    delta = F.col("leitura_atual") - F.col("leitura_anterior")
    sem_anterior = F.col("leitura_anterior").isNull()
    regras = {
        "consumo_nulo": F.col("consumo_kwh").isNull(),
        "troca_sem_leitura_regressiva": (F.col("origem_kwh") == "leitura_atual_troca_medidor") & ~(sem_anterior | (delta < 0)),
        "sinal_invertido_fora_do_padrao": (F.col("origem_kwh") == "consumo_absoluto")
            & ~(sem_anterior | ((delta > 0) & (F.abs(F.col("consumo_kwh") + delta) <= 1))),
        "kwh_faturado_negativo": F.col("kwh_faturado") < 0,
        "leitura_sem_medidor_no_cadastro": F.col("tipo_medidor").isNull(),
        "dias_faturados_nao_positivo": F.col("dias_faturados") <= 0,
    }
    fora = leituras.filter(F.greatest(*[F.coalesce(c, F.lit(False)) for c in regras.values()]))
    if fora.limit(1).count():
        (fora.withColumn("padrao", F.when(sem_anterior, "sem_anterior").when(delta < 0, "regressiva")
                                    .when(delta > 0, "atual_maior").otherwise("atual_igual"))
             .groupBy("origem_kwh", "padrao", F.substring("mes_referencia", 1, 4).alias("ano"))
             .count().orderBy(F.desc("count")).show(truncate=False))
    exigir_zero(leituras, regras)

    CACHE["energia"] = leituras.select(
        F.col("id_uc").alias("cod_uc"), F.col("mes_referencia").alias("competencia"),
        "kwh_faturado", F.col("consumo_kwh").alias("kwh_leitura"), "origem_kwh",
        F.lower("tipo_leitura").alias("tipo_leitura"), "tipo_medidor", "dias_faturados")
    return CACHE["energia"]

## 2. gold_dim_uc
`silver.unidades_consumidoras` + `silver.geracao_distribuida` + energia faturada → `gold.gold_dim_uc`. 1 linha por UC.

**Colunas**
- `classe` = `subgrupo_tarifario` (B1, B2, B3, A4); `classe_cadastro` = `classe` da Silver (opção B da DEC-023).
- `cod_alimentador` = `alimentador`; `possui_gd` = a UC está na Silver de GD.
- `consumo_medio_kwh_12m`: média do `kwh_faturado` (não do `consumo_kwh`, que tem negativos) nas competências da UC dentro das
  12 últimas competências presentes nos dados. Nula quando a UC não tem leitura nessa janela (ex.: UCs sem medidor, UC-12).
- `titular_mascarado`: iniciais do nome, sem partículas (`DA`, `DE`, `DO`, `DAS`, `DOS`, `E`), ex.: `M.S.`. O nome completo não sai daqui.
- `documento_mascarado`: só os 3 últimos dígitos, o resto como `*` (o tamanho distingue CPF de CNPJ). O documento completo não sai daqui.
- `potencia_kwp`: potência da Silver de GD. **Registro com `flag_potencia_anomala` (GD-02) é convertido de Wp para kWp (÷ 1000)
  quando o dado confirma**: o valor convertido cai na faixa das UCs com GD normais, e a energia injetada por kWp também. Se não
  confirma, fica vazio (a origem não diz qual é a potência). Convertidas e vazias saem no resultado (DEC-030).
- `data_ligacao` (cliente desde; a UC conta no `n_uc` a partir desse mês, DEC-030); `data_conexao_gd`, `modalidade_gd` e
  `energia_injetada_kwh_mes` da Silver de GD, vazios sem GD (DEC-027).

**Travas:** `cod_uc` único e igual ao cadastro da Silver; uma linha com GD por linha da Silver de GD; toda UC com GD tem data de
conexão e modalidade, e nenhuma UC sem GD tem dado de GD; potência só em UC com GD; toda potência anômala ou convertida com confirmação ou
vazia; documento mascarado com no máximo 3 dígitos à
mostra; contrato (tipos, nulos, domínios).

In [0]:
PARTICULAS = ["DA", "DE", "DO", "DAS", "DOS", "E"]
MODALIDADE_GD = {"GERACAO NA PROPRIA UC": "geracao_propria_uc", "AUTOCONSUMO REMOTO": "autoconsumo_remoto",
                 "GERACAO COMPARTILHADA": "geracao_compartilhada"}


def construir_dim_uc():
    u = spark.table(f"{SILVER}.unidades_consumidoras")
    modalidade = F.create_map(*[F.lit(x) for kv in MODALIDADE_GD.items() for x in kv])
    gd = (spark.table(f"{SILVER}.geracao_distribuida")
        .select("id_uc", "potencia_kwp", "flag_potencia_anomala", "data_conexao",
                modalidade[F.col("modalidade")].alias("modalidade_gd"),
                F.col("energia_injetada_kwh_media_mensal").alias("energia_injetada_kwh_mes"))
        .withColumn("_tem_gd", F.lit(True)))
    normal = gd.filter(~F.col("flag_potencia_anomala")).agg(                                     # GD-02: faixa das UCs normais
        F.min("potencia_kwp").alias("pot_min"), F.max("potencia_kwp").alias("pot_max"),
        F.min(F.col("energia_injetada_kwh_mes") / F.col("potencia_kwp")).alias("inj_min"),
        F.max(F.col("energia_injetada_kwh_mes") / F.col("potencia_kwp")).alias("inj_max")).first()
    convertida = F.col("potencia_kwp") / 1000
    wp_confirmado = (convertida.between(normal["pot_min"], normal["pot_max"])
                     & (F.col("energia_injetada_kwh_mes") / convertida).between(normal["inj_min"], normal["inj_max"]))
    gd = (gd.withColumn("_potencia_convertida", F.col("flag_potencia_anomala") & wp_confirmado)
            .withColumn("potencia_kwp", F.when(~F.col("flag_potencia_anomala"), F.col("potencia_kwp"))
                                         .when(F.col("_potencia_convertida"), convertida)))

    e = energia_faturada()
    ultima = e.agg(F.max("competencia")).first()[0]
    inicio = F.date_format(F.add_months(F.to_date(F.lit(f"{ultima}-01")), -11), "yyyy-MM")
    media = (e.filter(F.col("competencia") >= inicio)
        .groupBy(F.col("cod_uc").alias("id_uc")).agg(F.avg("kwh_faturado").alias("consumo_medio_kwh_12m")))

    palavras = F.filter(F.split(F.trim("nome_titular"), r"\s+"), lambda p: ~p.isin(*PARTICULAS))
    iniciais = F.array_join(F.transform(palavras, lambda p: F.concat(F.substring(p, 1, 1), F.lit("."))), "")
    documento = F.expr("concat(repeat('*', length(cpf_cnpj) - 3), right(cpf_cnpj, 3))")

    df = (u.join(gd, "id_uc", "left").join(media, "id_uc", "left")
        .select(
            F.col("id_uc").alias("cod_uc"),
            "municipio",
            "bairro",
            F.col("subgrupo_tarifario").alias("classe"),
            F.col("classe").alias("classe_cadastro"),
            F.col("alimentador").alias("cod_alimentador"),
            F.coalesce("_tem_gd", F.lit(False)).alias("possui_gd"),
            "consumo_medio_kwh_12m",
            F.when(F.size(palavras) > 0, iniciais).alias("titular_mascarado"),
            F.when(F.length("cpf_cnpj") > 3, documento).alias("documento_mascarado"),
            "potencia_kwp",
            "_potencia_convertida",
            "flag_potencia_anomala",
            "data_ligacao",
            F.col("data_conexao").alias("data_conexao_gd"),
            "modalidade_gd",
            "energia_injetada_kwh_mes"))

    total = exigir_chave_unica(df, ["cod_uc"])
    exigir_mesma_contagem(silver=u.count(), gold=total)
    exigir_mesma_contagem(silver_gd=gd.count(), gold_com_gd=df.filter("possui_gd").count())
    anomala = F.coalesce(F.col("flag_potencia_anomala"), F.lit(False))
    convertidas = df.filter(F.coalesce(F.col("_potencia_convertida"), F.lit(False))).count()
    anomalas_sem_potencia = df.filter(anomala & F.col("potencia_kwp").isNull()).count()
    exigir_mesma_contagem(anomalas_silver=gd.filter("flag_potencia_anomala").count(),
                          convertidas_mais_vazias=convertidas + anomalas_sem_potencia)
    df = df.drop("_potencia_convertida", "flag_potencia_anomala")
    exigir_zero(df, {
        "gd_sem_conexao_ou_modalidade": F.col("possui_gd") & (F.col("data_conexao_gd").isNull() | F.col("modalidade_gd").isNull()),
        "dado_de_gd_sem_gd": ~F.col("possui_gd") & (F.col("data_conexao_gd").isNotNull() | F.col("energia_injetada_kwh_mes").isNotNull()),
        "potencia_sem_gd": ~F.col("possui_gd") & F.col("potencia_kwp").isNotNull(),
        "documento_com_mais_de_3_digitos": F.length(F.regexp_replace("documento_mascarado", r"[^0-9]", "")) > 3,
    })
    df = aplicar_contrato(df, "gold_dim_uc")
    info = {
        "linhas": total,
        "com_gd": df.filter("possui_gd").count(),
        "sem_consumo_medio": df.filter(F.col("consumo_medio_kwh_12m").isNull()).count(),
        "gd_sem_potencia": df.filter(F.col("possui_gd") & F.col("potencia_kwp").isNull()).count(),
        "potencia_convertida_de_wp": convertidas,
        "potencia_anomala_sem_confirmacao": anomalas_sem_potencia,
        "janela_consumo_medio": f"12 competências até {ultima}",
    }
    return df, info


executar("gold_dim_uc", construir_dim_uc)

## 3. gold_fato_faturamento_mensal
`silver.faturas` (sem as de UC fora do cadastro, `flag_uc_orfa`) + energia faturada → `gold.gold_fato_faturamento_mensal`.
Grão UC × competência: uma UC tem uma única fatura por competência (conferido na Silver), e cada fatura tem exatamente uma leitura.

**Colunas**
- `receita_faturada` = `valor_total`; `receita_energia` = `valor_energia`; `tributos` = `tributos` (DEC-023).
- `receita_arrecadada` = `valor_total` se a fatura foi paga, `0` se não foi (não pagar é um fato, como na amostra do backend).
- `tarifa_media` = `receita_energia ÷ kwh_faturado`, nula quando `kwh_faturado = 0`.
- `bandeira`: `Verde` → `verde`, `Vermelha P1` → `vermelha_p1`...
- `status_pagamento` contra a **data de corte** (maior data de emissão ou pagamento entre todas as faturas, inclusive as órfãs):
  `paga` (até o vencimento), `paga_com_atraso`, `em_aberto` (não paga, vence no corte ou depois) e `em_atraso` (não paga, vencida).
- `dias_em_atraso`: pagamento − vencimento (paga com atraso) ou corte − vencimento (em atraso); nulo nos demais.
- `receita_bandeira` = `kwh_faturado × acréscimo da bandeira` (DEC-029); `tipo_leitura`, `tipo_medidor` e `dias_faturados` da
  leitura do mês (DEC-027).

**Travas:** faturas = leituras = linhas do fato (nenhuma fatura sem leitura nem o contrário); nenhuma fatura de UC sem medidor (UC-12);
chave única; `receita_faturada = receita_energia + tributos`; `receita_bandeira` entre zero e a receita de energia, e zero na bandeira
verde; situação coerente com a da Silver; `data_pagamento` vazia se e somente se não paga; `dias_em_atraso` vazio quando paga em dia
ou em aberto; nenhum `cod_uc` fora da `gold_dim_uc`; contrato.

In [0]:
BANDEIRA_GOLD = {"Verde": "verde", "Amarela": "amarela", "Vermelha P1": "vermelha_p1", "Vermelha P2": "vermelha_p2"}
SITUACAO_SILVER = {"paga": "PAGA", "paga_com_atraso": "PAGA COM ATRASO", "em_aberto": "EM ABERTO", "em_atraso": "EM ABERTO"}


def construir_fato_faturamento():
    todas = spark.table(f"{SILVER}.faturas")
    corte = todas.agg(F.max(F.greatest("data_emissao", "data_pagamento"))).first()[0]

    f = (todas.filter(~F.col("flag_uc_orfa"))
        .withColumnRenamed("id_uc", "cod_uc").withColumnRenamed("mes_referencia", "competencia"))
    e = energia_faturada()
    df = f.join(e, ["cod_uc", "competencia"], "inner")
    exigir_mesma_contagem(faturas=f.count(), leituras=e.count(), fato=df.count())

    sem_medidor = (spark.table(f"{SILVER}.unidades_consumidoras")
        .filter("flag_documento_repetido").select(F.col("id_uc").alias("cod_uc")))
    faturas_sem_medidor = f.join(sem_medidor, "cod_uc").count()
    assert faturas_sem_medidor == 0, f"{faturas_sem_medidor:,} faturas de UCs sem medidor (UC-12)"

    venc, pag, data_corte = F.col("data_vencimento"), F.col("data_pagamento"), F.lit(corte)
    status = (F.when(pag.isNotNull() & (pag <= venc), "paga")
               .when(pag.isNotNull(), "paga_com_atraso")
               .when(venc >= data_corte, "em_aberto")
               .otherwise("em_atraso"))
    mapa_bandeira = F.create_map(*[F.lit(x) for kv in BANDEIRA_GOLD.items() for x in kv])
    mapa_situacao = F.create_map(*[F.lit(x) for kv in SITUACAO_SILVER.items() for x in kv])
    acrescimo = F.create_map(*[F.lit(x) for kv in ACRESCIMO_BANDEIRA.items() for x in kv])

    df = (df
        .withColumn("_status_silver", F.col("status_pagamento"))
        .withColumn("status_pagamento", status)
        .withColumn("dias_em_atraso", F.when(F.col("status_pagamento") == "paga_com_atraso", F.datediff(pag, venc))
                                       .when(F.col("status_pagamento") == "em_atraso", F.datediff(data_corte, venc))))
    exigir_zero(df, {
        "situacao_diferente_da_silver": mapa_situacao[F.col("status_pagamento")] != F.col("_status_silver"),
        "bandeira_fora_do_mapa": F.col("bandeira_tarifaria").isNotNull() & mapa_bandeira[F.col("bandeira_tarifaria")].isNull(),
    })

    df = df.select(
        "cod_uc",
        "competencia",
        "kwh_faturado",
        "kwh_leitura",
        "origem_kwh",
        F.col("valor_total").alias("receita_faturada"),
        F.col("valor_energia").alias("receita_energia"),
        "tributos",
        F.when(pag.isNotNull(), F.col("valor_total")).otherwise(F.lit(0)).alias("receita_arrecadada"),
        F.when(F.col("kwh_faturado") != 0, F.col("valor_energia") / F.col("kwh_faturado")).alias("tarifa_media"),
        mapa_bandeira[F.col("bandeira_tarifaria")].alias("bandeira"),
        "data_vencimento",
        "data_pagamento",
        "status_pagamento",
        "dias_em_atraso",
        F.round(F.col("kwh_faturado") * acrescimo[mapa_bandeira[F.col("bandeira_tarifaria")]], 2).alias("receita_bandeira"),
        "tipo_leitura",
        "tipo_medidor",
        "dias_faturados")

    total = exigir_chave_unica(df, ["cod_uc", "competencia"])
    nao_paga = F.col("status_pagamento").isin("em_aberto", "em_atraso")
    exigir_zero(df, {
        "total_diferente_de_energia_mais_tributos":
            F.abs(F.col("receita_faturada") - F.col("receita_energia") - F.col("tributos")) > 0.01,
        "bandeira_maior_que_a_energia": (F.col("receita_bandeira") < 0) | (F.col("receita_bandeira") > F.col("receita_energia")),
        "bandeira_verde_com_acrescimo": (F.col("bandeira") == "verde") & (F.col("receita_bandeira") != 0),
        "pagamento_x_situacao": F.col("data_pagamento").isNull() != nao_paga,
        "dias_em_atraso_indevido": F.col("status_pagamento").isin("paga", "em_aberto") & F.col("dias_em_atraso").isNotNull(),
        "dias_em_atraso_ausente": F.col("status_pagamento").isin("paga_com_atraso", "em_atraso") & F.col("dias_em_atraso").isNull(),
    })
    exigir_sem_orfaos(df, "gold_fato_faturamento_mensal")
    df = aplicar_contrato(df, "gold_fato_faturamento_mensal")
    return df, {"linhas": total, "data_corte": str(corte), **contar_por(df, "origem_kwh"), **contar_por(df, "status_pagamento")}


executar("gold_fato_faturamento_mensal", construir_fato_faturamento)

## 4. gold_reconciliacao_faturamento
`silver.faturas` (todas, inclusive as de UC fora do cadastro) + `gold_fato_faturamento_mensal` → 1 linha por competência (DEC-023).
É a ponte do número bruto até o usado nos indicadores: quanto da receita vem de faturas sem cadastro (FAT-05) e quanto da energia
vem das correções de troca de medidor e de sinal invertido.

**Travas:** total = cadastro + órfãs; cadastro = energia + tributos; energia final = leitura + correções (0,001 kWh); receita do
cadastro, energia, tributos e kWh faturado iguais à soma do fato por competência; contrato.

In [0]:
def so_cadastro(c):
    return F.sum(F.when(~F.col("flag_uc_orfa"), F.col(c)).otherwise(F.lit(0)))


def correcao(origem):
    return F.sum(F.when(F.col("origem_kwh") == origem, F.col("kwh_faturado") - F.col("kwh_leitura")).otherwise(F.lit(0)))


def construir_reconciliacao():
    orfa = F.col("flag_uc_orfa")
    receita = (spark.table(f"{SILVER}.faturas")
        .groupBy(F.col("mes_referencia").alias("competencia"))
        .agg(F.sum("valor_total").alias("receita_faturada_total"),
             F.sum(F.when(orfa, F.col("valor_total")).otherwise(F.lit(0))).alias("receita_faturada_orfas"),
             so_cadastro("valor_total").alias("receita_faturada_cadastro"),
             so_cadastro("tributos").alias("tributos_cadastro"),
             so_cadastro("valor_energia").alias("receita_energia_cadastro"),
             F.sum(orfa.cast("int")).alias("n_faturas_orfas")))

    fato = obter("gold_fato_faturamento_mensal")
    energia = (fato.groupBy("competencia")
        .agg(F.sum("kwh_leitura").alias("kwh_leitura_total"),
             correcao("leitura_atual_troca_medidor").alias("kwh_correcao_troca_medidor"),
             correcao("consumo_absoluto").alias("kwh_correcao_sinal_invertido"),
             F.sum("kwh_faturado").alias("kwh_faturado_total"),
             F.sum("receita_faturada").alias("_fato_receita"),
             F.sum("receita_energia").alias("_fato_energia"),
             F.sum("tributos").alias("_fato_tributos")))

    df = receita.join(energia, "competencia", "full")
    total = exigir_chave_unica(df, ["competencia"])
    exigir_zero(df, {
        "competencia_so_de_um_lado": F.col("receita_faturada_total").isNull() | F.col("kwh_faturado_total").isNull(),
        "total_diferente_de_cadastro_mais_orfas":
            F.abs(F.col("receita_faturada_total") - F.col("receita_faturada_cadastro") - F.col("receita_faturada_orfas")) > 0.01,
        "cadastro_diferente_de_energia_mais_tributos":
            F.abs(F.col("receita_faturada_cadastro") - F.col("receita_energia_cadastro") - F.col("tributos_cadastro")) > 0.01,
        "energia_final_diferente_da_ponte": F.abs(F.col("kwh_faturado_total") - F.col("kwh_leitura_total")
            - F.col("kwh_correcao_troca_medidor") - F.col("kwh_correcao_sinal_invertido")) > 0.001,
        "cadastro_diferente_do_fato": (F.col("receita_faturada_cadastro") != F.col("_fato_receita"))
            | (F.col("receita_energia_cadastro") != F.col("_fato_energia")) | (F.col("tributos_cadastro") != F.col("_fato_tributos")),
    })
    df = aplicar_contrato(df.drop("_fato_receita", "_fato_energia", "_fato_tributos"), "gold_reconciliacao_faturamento")
    return df, {"linhas": total}


executar("gold_reconciliacao_faturamento", construir_reconciliacao)

## 5. Base UC × competência (continuidade e KPIs por recorte)
Uma linha por UC ligada em cada competência: é daqui que saem o `n_uc` (estoque) e os componentes somáveis das seções 6 e 7.
Montar a partir do cadastro, e não dos eventos, garante linha para todo recorte e mês, inclusive sem interrupção nem chamado,
e assim o denominador do DEC e do FEC não encolhe.

- Competências: as do fato; precisam ser contínuas.
- UC ligada na competência: `data_ligacao` até o mês. A base guarda o UC-mês ligado com fatura ou com algum evento (interrupção ou
  chamado). `n_uc` = UCs **faturadas** no mês, como no PRODIST (DEC-023, DEC-030): ficam fora as UCs da UC-12, que nunca são
  faturadas, e os meses de ligação antes da primeira fatura. Eventos em mês sem fatura continuam no numerador (contados no resultado).
- `possui_gd` por competência: `data_conexao` da GD até o mês (DEC-023).
- Interrupções: cada ocorrência é uma UC interrompida; entra no mês da abertura; horas = `duracao_min ÷ 60`. O alimentador vem do
  cadastro da UC.
- Chamados: pelo mês de abertura; nota de 1 a 5; chamado sem nota fica fora de `soma_notas` e `n_notas` (ATD-01).
- Medidor vigente no mês: instalado até o mês e não retirado até ele (no mês da troca, vale o novo); `sem_medidor` quando não há
  (UCs da UC-12). Faturas: contagem, não pagas na data de corte, valor não pago, pagas com atraso, dias de atraso e receita de
  bandeira; leituras: contagem e estimadas (DEC-027).

- Ocorrência registrada antes da ligação da UC (achado da Gold: existe em UCs ligadas durante o período) fica **fora** da Gold: a UC
  ainda não existia, então não entra no DEC/FEC dela nem do alimentador. A quantidade sai no resultado da continuidade, e a trava
  confere que Gold + excluídas = Silver (decisão desta cópia; a comparar com as outras).

**Travas:** competências contínuas; nenhuma UC sem `data_ligacao`; no máximo um medidor vigente por UC e mês; nenhuma fatura ou
chamado de UC em mês em que ela ainda não estava ligada (seriam perdidos em silêncio).

In [0]:
COMPONENTES = ["kwh_faturado", "receita_faturada", "receita_energia", "tributos", "receita_arrecadada",
               "horas_interrompidas", "n_interrupcoes", "n_chamados", "soma_notas", "n_notas", "receita_bandeira",
               "n_faturas", "n_faturas_nao_pagas", "valor_nao_pago", "n_faturas_pagas_com_atraso", "soma_dias_em_atraso",
               "n_leituras", "n_leituras_estimadas"]


def base_uc_mes():
    """UC × competência com os componentes somáveis, calculada uma vez por execução."""
    if "base" in CACHE:
        return CACHE["base"]

    fato = obter("gold_fato_faturamento_mensal")
    comp = fato.select("competencia").distinct()
    faixa = comp.agg(F.min("competencia").alias("ini"), F.max("competencia").alias("fim"), F.count("*").alias("n")).first()
    meses = F.months_between(F.to_date(F.lit(f"{faixa['fim']}-01")), F.to_date(F.lit(f"{faixa['ini']}-01")))
    esperados = spark.range(1).select(meses.cast("int")).first()[0] + 1
    assert faixa["n"] == esperados, f"competências não contínuas: {faixa['n']} de {esperados} entre {faixa['ini']} e {faixa['fim']}"

    u = spark.table(f"{SILVER}.unidades_consumidoras")
    exigir_zero(u, {"uc_sem_data_ligacao": F.col("data_ligacao").isNull()})
    gd = spark.table(f"{SILVER}.geracao_distribuida").select(
        F.col("id_uc").alias("cod_uc"), F.date_format("data_conexao", "yyyy-MM").alias("_mes_conexao"))
    ucs = (u.select(F.col("id_uc").alias("cod_uc"), F.date_format("data_ligacao", "yyyy-MM").alias("_mes_ligacao"))
        .join(obter("gold_dim_uc").select("cod_uc", "municipio", "bairro", "classe_cadastro", "classe", "cod_alimentador"), "cod_uc")
        .join(gd, "cod_uc", "left"))
    base = (comp.crossJoin(ucs).filter(F.col("_mes_ligacao") <= F.col("competencia"))
        .withColumn("possui_gd", F.coalesce(F.col("_mes_conexao") <= F.col("competencia"), F.lit(False)))
        .drop("_mes_ligacao", "_mes_conexao"))

    interrupcoes = (spark.table(f"{SILVER}.ocorrencias_rede")
        .groupBy(F.col("id_uc").alias("cod_uc"), F.date_format("data_abertura", "yyyy-MM").alias("competencia"))
        .agg(F.sum(F.col("duracao_min") / 60).alias("horas_interrompidas"), F.count("*").alias("n_interrupcoes")))
    chamados = (spark.table(f"{SILVER}.atendimentos")
        .groupBy(F.col("id_uc").alias("cod_uc"), F.date_format("data_abertura", "yyyy-MM").alias("competencia"))
        .agg(F.count("*").alias("n_chamados"), F.sum("nota_satisfacao").alias("soma_notas"), F.count("nota_satisfacao").alias("n_notas")))
    nao_paga = F.col("status_pagamento").isin("em_aberto", "em_atraso")
    faturas = fato.select(
        "cod_uc", "competencia", "kwh_faturado", "receita_faturada", "receita_energia", "tributos", "receita_arrecadada",
        "receita_bandeira",
        F.lit(1).alias("n_faturas"),
        nao_paga.cast("int").alias("n_faturas_nao_pagas"),
        F.when(nao_paga, F.col("receita_faturada")).otherwise(F.lit(0)).alias("valor_nao_pago"),
        (F.col("status_pagamento") == "paga_com_atraso").cast("int").alias("n_faturas_pagas_com_atraso"),
        F.coalesce("dias_em_atraso", F.lit(0)).alias("soma_dias_em_atraso"),
        F.lit(1).alias("n_leituras"),
        (F.col("tipo_leitura") == "estimada").cast("int").alias("n_leituras_estimadas"))

    medidores = spark.table(f"{SILVER}.medidores").select(
        F.col("id_uc").alias("cod_uc"), F.lower("tipo").alias("_tipo"),
        F.date_format("data_instalacao", "yyyy-MM").alias("_mes_instalacao"),
        F.date_format("data_retirada", "yyyy-MM").alias("_mes_retirada"))
    vigente = (base.select("cod_uc", "competencia").join(medidores, "cod_uc")
        .filter((F.col("_mes_instalacao") <= F.col("competencia"))
                & (F.col("_mes_retirada").isNull() | (F.col("_mes_retirada") > F.col("competencia"))))
        .groupBy("cod_uc", "competencia").agg(F.count("*").alias("_vigentes"), F.first("_tipo").alias("tipo_medidor")))
    exigir_zero(vigente, {"mais_de_um_medidor_vigente": F.col("_vigentes") > 1})
    base = (base.join(vigente.drop("_vigentes"), ["cod_uc", "competencia"], "left")
                .fillna("sem_medidor", ["tipo_medidor"]))

    chave = base.select("cod_uc", "competencia")
    perdidos = {nome: df.join(chave, ["cod_uc", "competencia"], "left_anti").count()
                for nome, df in {"faturas": faturas, "chamados": chamados}.items()}
    assert not any(perdidos.values()), f"eventos de UC ainda não ligada ou fora das competências: {perdidos}"
    antes = interrupcoes.join(chave, ["cod_uc", "competencia"], "left_anti").agg(
        F.coalesce(F.sum("n_interrupcoes"), F.lit(0)).alias("n"), F.coalesce(F.sum("horas_interrompidas"), F.lit(0)).alias("h")).first()
    CACHE["interrupcoes_antes_da_ligacao"] = antes

    base = (base.join(faturas, ["cod_uc", "competencia"], "left")
                .join(interrupcoes, ["cod_uc", "competencia"], "left")
                .join(chamados, ["cod_uc", "competencia"], "left")
                .fillna(0, COMPONENTES)
                .filter((F.col("n_faturas") > 0) | (F.col("n_interrupcoes") > 0) | (F.col("n_chamados") > 0)))
    CACHE["eventos_sem_fatura"] = base.filter(F.col("n_faturas") == 0).agg(
        F.coalesce(F.sum("n_interrupcoes"), F.lit(0)).alias("interrupcoes"),
        F.coalesce(F.sum("n_chamados"), F.lit(0)).alias("chamados")).first().asDict()
    CACHE["base"] = base
    return base

## 6. gold_continuidade_alimentador_mensal
Base UC × competência → alimentador × competência. `n_uc` = UCs faturadas atendidas pelo alimentador no mês (não só as afetadas);
`dec = horas_interrompidas ÷ n_uc` e `fec = n_interrupcoes ÷ n_uc` (DEC-014). Todo alimentador tem linha em todo mês desde a ligação
da sua primeira UC, com zero quando não houve interrupção.

**Travas:** chave única; horas e interrupções iguais às da Silver de ocorrências; `n_uc` > 0; contrato.

In [0]:
def construir_continuidade():
    base = base_uc_mes()
    df = (base.groupBy("cod_alimentador", "competencia")
        .agg(F.sum("n_faturas").alias("n_uc"), F.sum("n_interrupcoes").alias("n_interrupcoes"),
             F.sum("horas_interrompidas").alias("horas_interrompidas"))
        .withColumn("dec", F.col("horas_interrompidas") / F.col("n_uc"))
        .withColumn("fec", F.col("n_interrupcoes") / F.col("n_uc")))

    total = exigir_chave_unica(df, ["cod_alimentador", "competencia"])
    oc = spark.table(f"{SILVER}.ocorrencias_rede").agg(F.count("*").alias("n"), F.sum(F.col("duracao_min") / 60).alias("h")).first()
    antes = CACHE["interrupcoes_antes_da_ligacao"]
    gold = df.agg(F.sum("n_interrupcoes").alias("n"), F.sum("horas_interrompidas").alias("h")).first()
    assert gold["n"] + antes["n"] == oc["n"] and abs(float(gold["h"]) + float(antes["h"]) - float(oc["h"])) < 1e-6, \
        f"interrupções diferentes da Silver: gold {gold} + antes da ligação {antes} | silver {oc}"
    exigir_zero(df, {"n_uc_zero": F.col("n_uc") <= 0})
    df = aplicar_contrato(df, "gold_continuidade_alimentador_mensal")
    return df, {"linhas": total, "alimentadores": df.select("cod_alimentador").distinct().count(),
                "interrupcoes_antes_da_ligacao_excluidas": antes["n"],
                "interrupcoes_em_mes_sem_fatura": CACHE["eventos_sem_fatura"]["interrupcoes"]}


executar("gold_continuidade_alimentador_mensal", construir_continuidade)

## 7. gold_kpis_segmento_mensal
Base UC × competência → competência × município × bairro × classe do cadastro × `possui_gd` (por competência) × alimentador ×
tipo de medidor vigente. `receita_arrecadada` fica vazia no recorte sem fatura e `soma_notas` no recorte sem chamado avaliado
(DEC-030). Só componentes somáveis; DEC, FEC, satisfação, inadimplência e parte de leituras estimadas são calculados por
quem lê, sobre o filtro e o período escolhidos (DEC-023, DEC-027). `classe` acompanha a classe do cadastro.

**Travas:** chave única; `receita_faturada = receita_energia + tributos` por linha; por competência, kWh, receitas, bandeira e número
de faturas iguais ao fato; por competência e alimentador, `n_uc`, horas e interrupções iguais à continuidade; chamados iguais à Silver
de atendimentos; contrato.

In [0]:
def construir_kpis_segmento():
    base = base_uc_mes()
    chave = ["competencia", "municipio", "bairro", "classe_cadastro", "possui_gd", "cod_alimentador", "tipo_medidor"]
    df = base.groupBy(*chave).agg(F.first("classe").alias("classe"), F.countDistinct("classe").alias("_classes"),
                                  F.sum("n_faturas").alias("n_uc"), *[F.sum(c).alias(c) for c in COMPONENTES])
    # 3. vazio quando não há o que medir: sem fatura não há arrecadação; sem avaliação não há soma de notas (DEC-030)
    df = (df.withColumn("receita_arrecadada", F.when(F.col("n_faturas") > 0, F.col("receita_arrecadada")))
            .withColumn("soma_notas", F.when(F.col("n_notas") > 0, F.col("soma_notas"))))

    total = exigir_chave_unica(df, chave)
    exigir_zero(df, {
        "classe_cadastro_com_mais_de_uma_classe": F.col("_classes") > 1,
        "total_diferente_de_energia_mais_tributos":
            F.abs(F.col("receita_faturada") - F.col("receita_energia") - F.col("tributos")) > 0.01,
    })

    por_mes = df.groupBy("competencia").agg(*[F.sum(c).alias(c) for c in ["kwh_faturado", "receita_faturada",
                                                                         "receita_energia", "receita_bandeira", "n_faturas"]])
    fato = obter("gold_fato_faturamento_mensal").groupBy("competencia").agg(
        F.sum("kwh_faturado").alias("f_kwh"), F.sum("receita_faturada").alias("f_rf"), F.sum("receita_energia").alias("f_re"),
        F.sum("receita_bandeira").alias("f_rb"), F.count("*").alias("f_n"))
    conf_fato = por_mes.join(fato, "competencia", "full")
    exigir_zero(conf_fato, {
        "componentes_diferentes_do_fato": F.coalesce((F.col("kwh_faturado") != F.col("f_kwh"))
            | (F.col("receita_faturada") != F.col("f_rf")) | (F.col("receita_energia") != F.col("f_re"))
            | (F.col("receita_bandeira") != F.col("f_rb")) | (F.col("n_faturas") != F.col("f_n")), F.lit(True)),
    })
    por_alimentador = df.groupBy("competencia", "cod_alimentador").agg(
        F.sum("n_uc").alias("n_uc"), F.sum("horas_interrompidas").alias("h"), F.sum("n_interrupcoes").alias("n"))
    cont = obter("gold_continuidade_alimentador_mensal").select(
        "competencia", "cod_alimentador", F.col("n_uc").alias("c_uc"), F.col("horas_interrompidas").alias("c_h"),
        F.col("n_interrupcoes").alias("c_n"))
    exigir_zero(por_alimentador.join(cont, ["competencia", "cod_alimentador"], "full"), {
        "diferente_da_continuidade_por_alimentador": F.coalesce((F.col("n_uc") != F.col("c_uc"))
            | (F.col("n") != F.col("c_n")) | (F.abs(F.col("h") - F.col("c_h")) > 1e-6), F.lit(True)),
    })
    chamados = spark.table(f"{SILVER}.atendimentos").count()
    gold_chamados = df.agg(F.sum("n_chamados")).first()[0]
    assert gold_chamados == chamados, f"chamados: gold {gold_chamados:,} != silver {chamados:,}"

    df = aplicar_contrato(df.drop("_classes"), "gold_kpis_segmento_mensal")
    uc_meses = {f"uc_meses_{r['tipo_medidor']}": r["n"]
                for r in df.groupBy("tipo_medidor").agg(F.sum("n_uc").alias("n")).collect()}
    return df, {"linhas": total, **uc_meses}


executar("gold_kpis_segmento_mensal", construir_kpis_segmento)

## 8. gold_kpis_mensais
`gold_kpis_segmento_mensal` somada por competência: total da empresa. `dec = soma de horas ÷ soma de n_uc` e `fec` igual com as
interrupções; `satisfacao_media = soma das notas ÷ chamados com nota`. `n_chamados` só é vazio em mês sem atendimento carregado
(no dataset, os atendimentos cobrem todas as competências).

**Travas:** 1 linha por competência; mesmo número de competências do fato; contrato.

In [0]:
def construir_kpis_mensais():
    seg = obter("gold_kpis_segmento_mensal")
    cobertura = (spark.table(f"{SILVER}.atendimentos")
        .agg(F.date_format(F.min("data_abertura"), "yyyy-MM").alias("ini"), F.date_format(F.max("data_abertura"), "yyyy-MM").alias("fim"))
        .first())
    com_atendimento = F.col("competencia").between(cobertura["ini"], cobertura["fim"])
    df = (seg.groupBy("competencia")
        .agg(F.sum("kwh_faturado").alias("kwh_faturado"), F.sum("receita_faturada").alias("receita_faturada"),
             F.sum("receita_energia").alias("receita_energia"), F.sum("n_uc").alias("_n_uc"),
             F.sum("horas_interrompidas").alias("_horas"), F.sum("n_interrupcoes").alias("_interrupcoes"),
             F.sum("n_chamados").alias("_chamados"), F.sum("soma_notas").alias("_soma_notas"), F.sum("n_notas").alias("_n_notas"))
        .select("competencia", "kwh_faturado", "receita_faturada", "receita_energia",
                (F.col("_horas") / F.col("_n_uc")).alias("dec"),
                (F.col("_interrupcoes") / F.col("_n_uc")).alias("fec"),
                F.when(com_atendimento, F.col("_chamados")).alias("n_chamados"),
                F.when(com_atendimento & (F.col("_n_notas") > 0), F.col("_soma_notas") / F.col("_n_notas")).alias("satisfacao_media")))

    total = exigir_chave_unica(df, ["competencia"])
    exigir_mesma_contagem(competencias_fato=obter("gold_fato_faturamento_mensal").select("competencia").distinct().count(),
                          kpis=total)
    df = aplicar_contrato(df, "gold_kpis_mensais")
    return df, {"linhas": total, "atendimentos": f"{cobertura['ini']} a {cobertura['fim']}"}


executar("gold_kpis_mensais", construir_kpis_mensais)

## 9. gold_receita_decomposicao
Decomposição da variação da receita de energia (sem tributos, base provisória da DEC-023) entre dois períodos, por município ×
bairro × classe × `possui_gd`.

**Pares de períodos (DEC-028):** todos os pares de **mesmos meses em anos diferentes** com dado nos dois lados: cada mês de início,
cada duração de 1 a 12 meses e cada par de anos (ex.: `2024-01/2024-04` × `2025-01/2025-04`; `2022-07/2023-06` × `2025-07/2026-06`,
o par da pergunta da diretoria). Trechos diferentes (ex.: jan–abr × mar–jun) não são gravados: todas as combinações passariam de
1 bilhão de linhas. Se o portal precisar, o backend calcula na hora com a mesma fórmula, a partir do fato e de `receita_bandeira`.

**`possui_gd`:** estado no último mês do período atual, fixo para a UC nos dois períodos (DEC-023): quem conectou a GD entre eles
fica no grupo com GD dos dois lados, e o efeito GD é o efeito volume dessas linhas.

**Fórmulas (DEC-029)** (linha *i*, classe *c*, `Q` = kWh faturado, `R` = receita de energia, `B` = receita de bandeira,
`Rb = R − B`; 0 = base, 1 = atual):
- `P0 = R0/Q0` e `pb0 = Rb0/Q0` (na linha sem energia na base, os da classe no par); `β0 = P0 − pb0`.
- `efeito_mix = Q1_total × (Q1_c/Q1_total − Q0_c/Q0_total) × (Q0_i/Q0_c) × P0`: só a mudança de peso entre classes.
- `efeito_volume = (Q1 − Q0) × P0 − efeito_mix`.
- `efeito_tarifa = (Rb1 − Q1·pb0) − (Rb0 − Q0·pb0)`; `efeito_bandeira = (B1 − Q1·β0) − (B0 − Q0·β0)`.
- `efeito_preco = efeito_tarifa + efeito_bandeira`. Os quatro efeitos somam exatamente `R1 − R0`, por construção.

**Como roda:** o fato é somado primeiro por recorte, mês de conexão da GD e competência; cada competência é ligada aos pares em que
aparece (tabela pequena de pares × meses), e o cálculo é feito para todos os pares de uma vez, com janelas por par.

**Travas:** chave única; nenhuma fatura sem bandeira; `efeito_preco = tarifa + bandeira` (R$ 0,01) e
`preço + volume + mix = receita_atual − receita_base` (R$ 0,02) por linha; receitas de cada par iguais às do fato; contrato.

In [0]:
def pares_mesmos_meses(comps):
    # (periodo_base, periodo_atual, competência, lado, fim do atual) para todos os pares de mesmos meses em anos diferentes
    linhas, n = [], 0
    for duracao in range(1, 13):
        for i in range(len(comps)):
            for anos in range(1, 5):
                j = i + 12 * anos
                if j + duracao > len(comps) or i + duracao > len(comps):
                    continue
                base, atual = comps[i:i + duracao], comps[j:j + duracao]
                pb, pa = f"{base[0]}/{base[-1]}", f"{atual[0]}/{atual[-1]}"
                linhas += [(pb, pa, c, "0", atual[-1]) for c in base] + [(pb, pa, c, "1", atual[-1]) for c in atual]
                n += 1
    return spark.createDataFrame(linhas, "periodo_base string, periodo_atual string, competencia string, lado string, fim_atual string"), n


def construir_decomposicao():
    fato = obter("gold_fato_faturamento_mensal")
    comps = sorted(r[0] for r in fato.select("competencia").distinct().collect())
    pares, n_pares = pares_mesmos_meses(comps)
    assert n_pares > 0, f"nenhum par de mesmos meses em {len(comps)} competências"
    exigir_zero(fato, {"fatura_sem_bandeira": F.col("bandeira").isNull()})

    gd = spark.table(f"{SILVER}.geracao_distribuida").select(
        F.col("id_uc").alias("cod_uc"), F.date_format("data_conexao", "yyyy-MM").alias("_mes_conexao"))
    seg = ["municipio", "bairro", "classe"]
    mensal = (fato.join(obter("gold_dim_uc").select("cod_uc", *seg), "cod_uc").join(gd, "cod_uc", "left")
        .groupBy(*seg, "_mes_conexao", "competencia")
        .agg(F.sum(F.col("kwh_faturado").cast("double")).alias("q"),
             F.sum("receita_energia").alias("receita"),
             F.sum(F.col("receita_energia").cast("double")).alias("r"),
             F.sum(F.col("receita_bandeira").cast("double")).alias("b")))

    par = ["periodo_base", "periodo_atual"]
    p = (mensal.join(F.broadcast(pares), "competencia")
        .withColumn("possui_gd", F.coalesce(F.col("_mes_conexao") <= F.col("fim_atual"), F.lit(False)))
        .groupBy(*par, *seg, "possui_gd").pivot("lado", ["0", "1"])
        .agg(F.sum("q").alias("q"), F.sum("r").alias("r"), F.sum("b").alias("b"), F.sum("receita").alias("receita"))
        .fillna(0))

    todas, classe = Window.partitionBy(*par), Window.partitionBy(*par, "classe")
    q0c, q1c = F.sum("0_q").over(classe), F.sum("1_q").over(classe)
    q0t, q1t = F.sum("0_q").over(todas), F.sum("1_q").over(todas)
    tem_base = F.col("0_q") > 0
    rb0, rb1 = F.col("0_r") - F.col("0_b"), F.col("1_r") - F.col("1_b")
    p0 = F.when(tem_base, F.col("0_r") / F.col("0_q")).otherwise(F.sum("0_r").over(classe) / q0c)
    pb0 = F.when(tem_base, rb0 / F.col("0_q")).otherwise(F.sum(rb0).over(classe) / q0c)
    p = (p.withColumn("_p0", p0).withColumn("_pb0", pb0).withColumn("_b0", F.col("_p0") - F.col("_pb0"))
          .withColumn("_mix", q1t * (q1c / q1t - q0c / q0t) * (F.col("0_q") / q0c) * F.col("_p0")))

    df = p.select(
        *par, *seg, "possui_gd",
        F.col("0_receita").alias("receita_base"),
        F.col("1_receita").alias("receita_atual"),
        F.round((rb1 - F.col("1_q") * F.col("_pb0")) - (rb0 - F.col("0_q") * F.col("_pb0")), 2).alias("efeito_tarifa"),
        F.round((F.col("1_b") - F.col("1_q") * F.col("_b0")) - (F.col("0_b") - F.col("0_q") * F.col("_b0")), 2).alias("efeito_bandeira"),
        F.round((F.col("1_q") - F.col("0_q")) * F.col("_p0") - F.col("_mix"), 2).alias("efeito_volume"),
        F.round(F.col("_mix"), 2).alias("efeito_mix"),
    ).withColumn("efeito_preco", F.col("efeito_tarifa") + F.col("efeito_bandeira"))

    chave = [*par, *seg, "possui_gd"]
    total = exigir_chave_unica(df, chave)
    exigir_zero(df, {
        "preco_diferente_de_tarifa_mais_bandeira":
            F.abs(F.col("efeito_preco") - F.col("efeito_tarifa") - F.col("efeito_bandeira")) > 0.01,
        "efeitos_diferentes_da_variacao": F.abs(F.col("efeito_preco") + F.col("efeito_volume") + F.col("efeito_mix")
            - (F.col("receita_atual") - F.col("receita_base"))) > 0.02,
    })

    soma_fato = (fato.groupBy("competencia").agg(F.sum("receita_energia").alias("receita"))
        .join(F.broadcast(pares), "competencia").groupBy(*par).pivot("lado", ["0", "1"]).agg(F.sum("receita")))
    soma_gold = df.groupBy(*par).agg(F.sum("receita_base").alias("g0"), F.sum("receita_atual").alias("g1"))
    exigir_zero(soma_fato.join(soma_gold, par, "full"), {
        "receita_do_par_diferente_do_fato": F.coalesce((F.col("0") != F.col("g0")) | (F.col("1") != F.col("g1")), F.lit(True)),
    })

    diretoria = (f"{comps[0]}/{comps[11]}", f"{comps[-12]}/{comps[-1]}")
    s = (df.filter((F.col("periodo_base") == diretoria[0]) & (F.col("periodo_atual") == diretoria[1]))
        .agg(*[F.sum(c).alias(c) for c in ["receita_base", "receita_atual", "efeito_tarifa", "efeito_bandeira",
                                            "efeito_volume", "efeito_mix"]],
             F.sum(F.when(F.col("possui_gd"), F.col("efeito_volume"))).alias("efeito_volume_gd"))).first().asDict()
    print(f"par da diretoria {diretoria[0]} -> {diretoria[1]}:",
          {"variacao_%": round(float(s["receita_atual"] / s["receita_base"] - 1) * 100, 2),
           **{k: round(float(v), 2) for k, v in s.items() if k.startswith("efeito")}})

    df = aplicar_contrato(df, "gold_receita_decomposicao")
    return df, {"linhas": total, "pares": n_pares}


executar("gold_receita_decomposicao", construir_decomposicao)

## 10. gold_fila_suspeitas
UCs para a fila de inspeção, por prioridade (`score_suspeita` = posição, 1 = mais suspeita). Fora do agente (decisão de produto).

**Sinais** (calibrados nas inspeções de campo da Silver, que dizem quais UCs tinham irregularidade confirmada):
- Queda do consumo médio (`kwh_faturado`) das 12 últimas competências contra as 12 anteriores. É o sinal que separa as UCs com
  irregularidade confirmada das demais; medidor eletromecânico e leituras estimadas só ajudam a ordenar dentro do grupo com queda.
- Medidor ativo eletromecânico; parte das leituras das 12 últimas competências que foram estimadas.
- Consumo abaixo do bairro foi testado e descartado: quase não separa as inspeções confirmadas das sem irregularidade.

**Quem entra:** UC sem GD (a queda da GD é legítima), com pelo menos 6 meses em cada janela e queda de 15% ou mais.
**Quem sai:** UCs com irregularidade confirmada (já regularizadas) e UCs inspecionadas sem irregularidade nos 12 meses antes do fim
dos dados; inspecionada sem irregularidade há mais tempo pode voltar (DEC-030).

**Validação:** a regra de entrada é medida nas UCs inspecionadas com a janela na **data de cada inspeção** (12 competências antes
dela × 12 anteriores), que é o que a equipe via ao decidir; a janela atual inclui meses depois da regularização. Precisão e cobertura
saem no resultado.

**Ordem:** pontos (queda ≥ 30%: 3; queda de 15% a 30%: 1; medidor eletromecânico: 1; 25% ou mais de leituras estimadas: 1),
depois maior `kwh_recuperavel_est`. `motivos` lista os sinais em texto. `kwh_recuperavel_est` = média anterior − média recente,
em kWh/mês.

**Travas:** chave única; `score_suspeita` de 1 a N sem repetição; todo suspeito com pelo menos um motivo; nenhum `cod_uc` fora da
`gold_dim_uc`; contrato.

In [0]:
QUEDA_ENTRADA, QUEDA_FORTE, ESTIMADAS_ALTA, MESES_MINIMOS = 0.15, 0.30, 0.25, 6


def em(janela):
    return F.col("competencia").isin(janela)


def validar_na_data_da_inspecao(fato, inspecao):
    """Precisão e cobertura da regra de entrada com a janela que a equipe via: 12 competências antes da inspeção x 12 anteriores."""
    gd = obter("gold_dim_uc").filter("possui_gd").select("cod_uc")
    distancia = F.months_between(F.to_date(F.concat("_mes_inspecao", F.lit("-01"))),
                                 F.to_date(F.concat("competencia", F.lit("-01")))).cast("int")
    janela = (fato.select("cod_uc", "competencia", "kwh_faturado").join(inspecao, "cod_uc").join(gd, "cod_uc", "left_anti")
        .withColumn("_d", distancia)
        .groupBy("cod_uc", "_resultado")
        .agg(F.avg(F.when(F.col("_d").between(1, 12), F.col("kwh_faturado"))).alias("_recente"),
             F.avg(F.when(F.col("_d").between(13, 24), F.col("kwh_faturado"))).alias("_anterior"),
             F.count(F.when(F.col("_d").between(1, 12), 1)).alias("_n_recente"),
             F.count(F.when(F.col("_d").between(13, 24), 1)).alias("_n_anterior"))
        .filter((F.col("_n_recente") >= MESES_MINIMOS) & (F.col("_n_anterior") >= MESES_MINIMOS) & (F.col("_anterior") > 0)))
    candidato = (1 - F.col("_recente") / F.col("_anterior")) >= QUEDA_ENTRADA
    confirmada = F.col("_resultado") == "IRREGULARIDADE CONFIRMADA"
    v = janela.agg(F.sum((candidato & confirmada).cast("int")).alias("acertos"),
                   F.sum(candidato.cast("int")).alias("apontadas"),
                   F.sum(confirmada.cast("int")).alias("confirmadas"),
                   F.count("*").alias("inspecionadas_medidas")).first()
    return {"precisao_na_data_da_inspecao": round(v["acertos"] / v["apontadas"], 3) if v["apontadas"] else None,
            "cobertura_na_data_da_inspecao": round(v["acertos"] / v["confirmadas"], 3) if v["confirmadas"] else None,
            "inspecionadas_medidas": v["inspecionadas_medidas"]}


def construir_fila_suspeitas():
    fato = obter("gold_fato_faturamento_mensal")
    comps = sorted(r[0] for r in fato.select("competencia").distinct().collect())
    recentes, anteriores = comps[-12:], comps[-24:-12]

    consumo = fato.groupBy("cod_uc").agg(
        F.avg(F.when(em(recentes), F.col("kwh_faturado"))).alias("_recente"),
        F.avg(F.when(em(anteriores), F.col("kwh_faturado"))).alias("_anterior"),
        F.count(F.when(em(recentes), 1)).alias("_n_recente"),
        F.count(F.when(em(anteriores), 1)).alias("_n_anterior"))
    estimadas = (spark.table(f"{SILVER}.leituras").filter(F.col("mes_referencia").isin(recentes))
        .groupBy(F.col("id_uc").alias("cod_uc"))
        .agg(F.avg((F.col("tipo_leitura") == "ESTIMADA").cast("double")).alias("_estimadas")))
    medidor = (spark.table(f"{SILVER}.medidores").filter(F.col("status") == "ATIVO")
        .select(F.col("id_uc").alias("cod_uc"), (F.col("tipo") == "ELETROMECANICO").alias("_eletromecanico")))
    inspecao = spark.table(f"{SILVER}.inspecoes").select(
        F.col("id_uc").alias("cod_uc"), F.col("resultado").alias("_resultado"),
        F.date_format("data", "yyyy-MM").alias("_mes_inspecao"))

    base = (obter("gold_dim_uc").filter(~F.col("possui_gd")).select("cod_uc")
        .join(consumo, "cod_uc").join(estimadas, "cod_uc", "left").join(medidor, "cod_uc")
        .join(inspecao, "cod_uc", "left")
        .filter((F.col("_n_recente") >= MESES_MINIMOS) & (F.col("_n_anterior") >= MESES_MINIMOS) & (F.col("_anterior") > 0))
        .withColumn("_queda", (1 - F.col("_recente") / F.col("_anterior")).cast("double")))
    candidato = F.col("_queda") >= QUEDA_ENTRADA

    validacao = validar_na_data_da_inspecao(fato, inspecao)

    forte = F.col("_queda") >= QUEDA_FORTE
    eletromecanico = F.coalesce(F.col("_eletromecanico"), F.lit(False))
    muitas_estimadas = F.coalesce(F.col("_estimadas") >= ESTIMADAS_ALTA, F.lit(False))
    pontos = (F.when(forte, 3).otherwise(1) + eletromecanico.cast("int") + muitas_estimadas.cast("int"))
    motivos = F.filter(F.array(
        F.format_string("Queda de %.0f%% no consumo médio das 12 últimas competências contra as 12 anteriores", F.col("_queda") * 100),
        F.when(eletromecanico, F.lit("Medidor eletromecânico")),
        F.when(muitas_estimadas, F.format_string("%.0f%% das leituras das 12 últimas competências foram estimadas",
                                                  F.col("_estimadas") * 100))), lambda m: m.isNotNull())
    recuperavel = F.greatest(F.col("_anterior") - F.col("_recente"), F.lit(0))

    confirmada = F.col("_resultado") == "IRREGULARIDADE CONFIRMADA"
    recente = F.col("_mes_inspecao") >= F.lit(recentes[0])
    fora_da_fila = F.coalesce(confirmada | recente, F.lit(False))
    df = (base.filter(candidato & ~fora_da_fila)
        .withColumn("_pontos", pontos).withColumn("kwh_recuperavel_est", recuperavel)
        .withColumn("score_suspeita", F.row_number().over(
            Window.orderBy(F.desc("_pontos"), F.desc("kwh_recuperavel_est"), "cod_uc")))
        .select("cod_uc", "score_suspeita", motivos.alias("motivos"), "kwh_recuperavel_est"))

    total = exigir_chave_unica(df, ["cod_uc"])
    faixa = df.agg(F.min("score_suspeita").alias("min"), F.max("score_suspeita").alias("max"),
                   F.countDistinct("score_suspeita").alias("n")).first()
    assert total == 0 or (faixa["min"] == 1 and faixa["max"] == total == faixa["n"]), f"score_suspeita fora de 1..N: {faixa}"
    exigir_zero(df, {"sem_motivo": F.size("motivos") == 0, "recuperavel_negativo": F.col("kwh_recuperavel_est") < 0})
    exigir_sem_orfaos(df, "gold_fila_suspeitas")
    df = aplicar_contrato(df, "gold_fila_suspeitas")
    return df, {"linhas": total, "janela": f"{recentes[0]}/{recentes[-1]} contra {anteriores[0]}/{anteriores[-1]}", **validacao}


executar("gold_fila_suspeitas", construir_fila_suspeitas)

## 11. gold_medidor_uc
Medidor vigente de cada UC (o `ATIVO` da Silver, que garante no máximo um por UC), com idade e trocas (DEC-027). UCs sem medidor
(UC-12) não têm linha. `idade_anos` é medida no fim da última competência dos dados; `n_trocas` conta os medidores já retirados
da UC e `data_ultima_troca` é a retirada mais recente. A data da última manutenção não existe na origem (MED-06).

**Travas:** chave única; uma linha por UC com medidor ativo; idade não negativa; nenhum `cod_uc` fora da `gold_dim_uc`; contrato.

In [0]:
def construir_medidor_uc():
    fato = obter("gold_fato_faturamento_mensal")
    ultima = fato.agg(F.max("competencia")).first()[0]
    referencia = F.last_day(F.to_date(F.lit(f"{ultima}-01")))
    m = spark.table(f"{SILVER}.medidores")
    trocas = (m.filter(F.col("status") == "RETIRADO").groupBy(F.col("id_uc").alias("cod_uc"))
        .agg(F.count("*").alias("n_trocas"), F.max("data_retirada").alias("data_ultima_troca")))
    df = (m.filter(F.col("status") == "ATIVO")
        .select(F.col("id_uc").alias("cod_uc"), F.col("id_medidor").alias("cod_medidor"), F.lower("tipo").alias("tipo_medidor"),
                "modelo", "data_instalacao",
                (F.months_between(referencia, F.col("data_instalacao")) / 12).alias("idade_anos"))
        .join(trocas, "cod_uc", "left").fillna(0, ["n_trocas"]))

    total = exigir_chave_unica(df, ["cod_uc"])
    exigir_mesma_contagem(ucs_com_medidor_ativo=m.filter(F.col("status") == "ATIVO").select("id_uc").distinct().count(), gold=total)
    exigir_zero(df, {"idade_negativa": F.col("idade_anos") < 0})
    exigir_sem_orfaos(df, "gold_medidor_uc")
    df = aplicar_contrato(df, "gold_medidor_uc")
    return df, {"linhas": total, "referencia_idade": f"fim de {ultima}", **contar_por(df, "tipo_medidor")}


executar("gold_medidor_uc", construir_medidor_uc)

## 12. gold_interrupcoes_uc
Uma linha por ocorrência de rede, para a UC 360. O alimentador vem do cadastro da UC; `competencia` é o mês da abertura;
`duracao_horas = duracao_min ÷ 60`. Ocorrências antes do mês de ligação da UC ficam fora (OCO-04, DEC-030), como na continuidade.

**Travas:** chave única; Gold + excluídas antes da ligação = Silver; por competência e alimentador, horas e contagem iguais à
`gold_continuidade_alimentador_mensal`; nenhum `cod_uc` fora da `gold_dim_uc`; contrato.

In [0]:
def construir_interrupcoes_uc():
    dim = obter("gold_dim_uc").select("cod_uc", "cod_alimentador", F.date_format("data_ligacao", "yyyy-MM").alias("_mes_ligacao"))
    oc = spark.table(f"{SILVER}.ocorrencias_rede")
    df = (oc.select(F.col("id_ocorrencia").alias("cod_ocorrencia"), F.col("id_uc").alias("cod_uc"),
                    F.date_format("data_abertura", "yyyy-MM").alias("competencia"), "data_abertura", "data_encerramento",
                    (F.col("duracao_min") / 60).alias("duracao_horas"), "tipo", "causa")
        .join(dim, "cod_uc")
        .filter(F.col("competencia") >= F.col("_mes_ligacao")).drop("_mes_ligacao"))

    total = exigir_chave_unica(df, ["cod_ocorrencia"])
    excluidas = oc.count() - total
    assert excluidas == CACHE["interrupcoes_antes_da_ligacao"]["n"], \
        f"excluídas {excluidas:,} != antes da ligação {CACHE['interrupcoes_antes_da_ligacao']['n']:,}"
    por_alimentador = df.groupBy("competencia", "cod_alimentador").agg(F.count("*").alias("n"), F.sum("duracao_horas").alias("h"))
    cont = obter("gold_continuidade_alimentador_mensal").filter(F.col("n_interrupcoes") > 0).select(
        "competencia", "cod_alimentador", F.col("n_interrupcoes").alias("c_n"), F.col("horas_interrompidas").alias("c_h"))
    exigir_zero(por_alimentador.join(cont, ["competencia", "cod_alimentador"], "full"), {
        "diferente_da_continuidade": F.coalesce((F.col("n") != F.col("c_n")) | (F.abs(F.col("h") - F.col("c_h")) > 1e-6), F.lit(True)),
    })
    exigir_sem_orfaos(df, "gold_interrupcoes_uc")
    df = aplicar_contrato(df, "gold_interrupcoes_uc")
    return df, {"linhas": total, "excluidas_antes_da_ligacao": excluidas}


executar("gold_interrupcoes_uc", construir_interrupcoes_uc)

## 13. gold_atendimentos_uc
Uma linha por chamado, para a UC 360. `competencia` é o mês de abertura. O histórico de interações (texto livre) fica fora da Gold:
é dado sensível e não é usado pelo portal nem pelo agente (DEC-027).

**Travas:** chave única; mesma quantidade da Silver; nenhum chamado antes da ligação da UC; nenhum `cod_uc` fora da `gold_dim_uc`;
contrato.

In [0]:
def construir_atendimentos_uc():
    at = spark.table(f"{SILVER}.atendimentos")
    df = at.select("protocolo", F.col("id_uc").alias("cod_uc"), F.date_format("data_abertura", "yyyy-MM").alias("competencia"),
                   "data_abertura", "data_fechamento", "canal", "assunto", "subassunto", "nota_satisfacao")

    total = exigir_chave_unica(df, ["protocolo"])
    exigir_mesma_contagem(silver=at.count(), gold=total)
    ligacao = obter("gold_dim_uc").select("cod_uc", F.date_format("data_ligacao", "yyyy-MM").alias("_mes_ligacao"))
    exigir_zero(df.join(ligacao, "cod_uc", "left"), {"chamado_antes_da_ligacao": F.col("competencia") < F.col("_mes_ligacao")})
    exigir_sem_orfaos(df, "gold_atendimentos_uc")
    df = aplicar_contrato(df, "gold_atendimentos_uc")
    return df, {"linhas": total}


executar("gold_atendimentos_uc", construir_atendimentos_uc)

## 14. gold_inspecoes_uc
Uma linha por inspeção de campo: histórico na UC 360, taxa de irregularidade por recorte (H2) e régua da fila de suspeitas. Fora do
agente, como a fila. Vale a `data` da inspeção, não a aba de origem (INSP-05). O nome do inspetor fica fora da Gold (DEC-027).

**Travas:** chave única; mesma quantidade da Silver; resultado no domínio; nenhum `cod_uc` fora da `gold_dim_uc`; contrato.

In [0]:
RESULTADO_GOLD = {"IRREGULARIDADE CONFIRMADA": "irregularidade_confirmada", "SEM IRREGULARIDADE": "sem_irregularidade"}


def construir_inspecoes_uc():
    insp = spark.table(f"{SILVER}.inspecoes")
    resultado = F.create_map(*[F.lit(x) for kv in RESULTADO_GOLD.items() for x in kv])
    df = insp.select(F.col("id_inspecao").alias("cod_inspecao"), F.col("id_uc").alias("cod_uc"),
                     F.col("data").alias("data_inspecao"), F.date_format("data", "yyyy-MM").alias("competencia"),
                     resultado[F.col("resultado")].alias("resultado"), "tipo_irregularidade", "kwh_recuperado")

    total = exigir_chave_unica(df, ["cod_inspecao"])
    exigir_mesma_contagem(silver=insp.count(), gold=total)
    exigir_sem_orfaos(df, "gold_inspecoes_uc")
    df = aplicar_contrato(df, "gold_inspecoes_uc")
    return df, {"linhas": total, **contar_por(df, "resultado")}


executar("gold_inspecoes_uc", construir_inspecoes_uc)

## Resumo
Imprime o estado de cada tabela. Output **confidencial** (contagens reais): nunca no Git.
Falha se alguma tabela não foi executada ou falhou.

In [0]:
print("CONFIDENCIAL: resultados derivados do dataset. Não versionar.")
print(f"gravar = {'sim' if GRAVAR else 'nao'} | {datetime.now().isoformat(timespec='seconds')}")
incompletas = []
for nome in TABELAS_ESPERADAS:
    r = RESULTADOS.get(nome)
    if r is None:
        print(f"- {nome}: NÃO EXECUTADA")
        incompletas.append(nome)
        continue
    print(f"- {nome}: {r}")
    if r["status"] == "FALHA":
        incompletas.append(nome)

assert not incompletas, f"Gold NÃO concluída. Não executadas ou com falha: {incompletas}"
print("[OK] Todas as tabelas montadas e validadas" + (" e gravadas." if GRAVAR else " (nada foi gravado: gravar = nao)."))